# Lab 3 · The lie of random splits

**What you'll build:** spatially-blocked cross-validation, and the demonstration
of why the standard tutorial version would have handed you a fake result.

This is the most important notebook in the series. If a judge asks you one
methods question, it will probably be this one.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labgrader import panel, grade_lab, TARGET, CLIMATE

df = panel()
print(df.shape, "rows x columns")
print("target column:", TARGET)

## 1. The standard advice, and why it can fail

Every ML tutorial says: split your data at random, train on 80%, test on 20%,
report the test score. That works when rows are *independent*.

Rows in spatial data usually are not. Two corridors 200 m apart share weather,
soil and neighbourhood, so a random split hands the model the answer from next
door and the test score measures **spatial autocorrelation**, not skill.

That is the textbook warning. Now do something the textbook doesn't: **check
whether it is actually true of your data**, instead of assuming it.

In [ ]:
# Is a corridor more like its nearest neighbour than like a random corridor?
from scipy.spatial import cKDTree

c = df.groupby("id").agg(x=("x_m", "first"), y=("y_m", "first"), t=(TARGET, "mean"))
tree = cKDTree(c[["x", "y"]].to_numpy())
_, idx = tree.query(c[["x", "y"]].to_numpy(), k=2)      # k=2: self, then neighbour
neigh = c["t"].to_numpy()[idx[:, 1]]

rng = np.random.default_rng(0)
rand = rng.permutation(c["t"].to_numpy())

print(f"corr(corridor, nearest neighbour) : {np.corrcoef(c['t'], neigh)[0,1]:+.3f}")
print(f"corr(corridor, random corridor)   : {np.corrcoef(c['t'], rand)[0,1]:+.3f}")

> **Not what the textbook predicted.** Both correlations are near zero, and the
> random pairing is if anything the higher of the two. Surrey's corridor stress
> is *not* meaningfully autocorrelated in space at this scale.
>
> That is not a broken measurement — it is Phase 3b's finding arriving early.
> Water stress here is driven by a corridor's own aspect, soil and canopy
> composition, which change over metres, not by anything that varies smoothly
> across a city. Neighbouring corridors are not alike.
>
> So the famous spatial-leakage argument is weak for this dataset. Keep going —
> there is a second dependence, and it is enormous.

In [ ]:
# Each corridor appears four times, once per summer. How alike are its own rows?
piv = df.pivot_table(index="id", columns="year", values=TARGET)
print("corr(2023, 2024), same corridor : %+.3f" % piv[2023].corr(piv[2024]))
print("corr(2022, 2025), same corridor : %+.3f" % piv[2022].corr(piv[2025]))

> **There it is: ~0.87.** The leak in this panel is not spatial, it is
> **repeated measures**. A random row-split puts corridor 36's 2023 row in
> training and its 2024 row in testing, and those two rows are nearly the same
> row. The model doesn't need to learn climate; it can recognise the corridor.
>
> The general lesson is the transferable one: **you do not assume which
> dependence you have, you measure it.** Someone who recited the spatial-leakage
> warning without checking would have described this dataset wrongly — and
> someone who saw the near-zero spatial correlation and concluded "so a random
> split is fine here" would have been wrong in the opposite direction.

## 2. The fix: block by *place*, hold out whole blocks

Cut the map into 5 geographic blocks with k-means on corridor coordinates.
Train on 4, test on the 5th. Rotate.

The critical detail — and the reason this design closes *both* dependences at
once — is that you **cluster the corridor, not the row**. Every summer of a
corridor inherits its corridor's block, so a corridor can never appear in both
train and test. Spatial blocking and repeated-measures grouping become the same
operation, for free.

`src/pipeline/experiment.py` says exactly this in its docstring: *"Corridor
grouping is automatic because blocking is done on the corridor, not the row."*
Given what you just measured, the grouping is doing most of the work here and
the spatial part is nearly free insurance. Both are worth having; only one of
them is load-bearing over Surrey.

### ✏️ Assignment 1 — `assign_blocks`

Return a block label **per row**, such that all four rows of any corridor share
a label.

1. `df.groupby("id")[["x_m", "y_m"]].first()` — one point per corridor.
2. `KMeans(n_clusters=n_blocks, random_state=seed, n_init=10).fit_predict(...)`.
3. Wrap the labels in a `pd.Series` indexed by corridor `id`.
4. `df["id"].map(that_series)` to expand back to rows.

In [ ]:
from sklearn.cluster import KMeans

def assign_blocks(df, n_blocks=5, seed=26910):
    """A spatial block label for every row. All rows of a corridor share one."""
    # >>> YOUR TURN
    raise NotImplementedError

In [ ]:
blocks = assign_blocks(df, 5, 26910)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["x_m"], df["y_m"], c=blocks, cmap="tab10", s=18)
ax.set_aspect("equal"); ax.set_xlabel("easting (m)"); ax.set_ylabel("northing (m)")
ax.set_title("Five spatial blocks. Each is held out whole, in turn.")
plt.show()

split = df.assign(b=blocks.to_numpy()).groupby("id")["b"].nunique()
print("corridors split across blocks:", int((split > 1).sum()), "(must be 0)")

### ✏️ Assignment 2 — `blocked_cv_rmse`

Leave-one-block-out. For each block: fit on everything else, predict the block,
collect the RMSE. Return the **mean over folds**.

Use `LinearRegression` — Lab 4 explains why the honest choice is ridge.

In [ ]:
from sklearn.linear_model import LinearRegression

def blocked_cv_rmse(df, features, target, blocks):
    """Mean RMSE over leave-one-block-out folds."""
    # >>> YOUR TURN
    raise NotImplementedError

## 3. The comparison

Now run the *same* model both ways: once with a random split, once blocked.

In [ ]:
FEATS = ["PPT_sm", "Tmax_sm", "CMD_sm"]

# Random folds: same number of folds, same model, assignment ignores geography.
rng = np.random.default_rng(26910)
random_blocks = pd.Series(rng.integers(0, 5, len(df)), index=df.index)

RANDOM_CV_RMSE  = blocked_cv_rmse(df, FEATS, TARGET, random_blocks)
BLOCKED_CV_RMSE = blocked_cv_rmse(df, FEATS, TARGET, blocks)

print(f"random  folds : RMSE {RANDOM_CV_RMSE:.5f}")
print(f"blocked folds : RMSE {BLOCKED_CV_RMSE:.5f}")
print(f"\ninflation from random splitting: {(1 - RANDOM_CV_RMSE/BLOCKED_CV_RMSE)*100:.1f}%")

> **The random-split number is better, and it is better because it cheated.**
> That gap is fake skill: it is the model recognising corridors it has already
> seen in another summer.
>
> Note the size, though — a couple of percent, not a transformation. Be honest
> about that rather than overselling it. The reason it is small is one you
> already know from Lab 1: **there is barely any skill here to inflate.** Both
> numbers are close to what you'd get predicting the mean, so leakage has little
> to work with.
>
> That is exactly why the discipline matters more, not less. On a dataset where
> the model *did* have skill, this same leak would inflate it dramatically — and
> you would have no way to tell from the score alone. You choose the fold
> structure before you see the result, or the result is not evidence.
>
> It is also why "why is your R² so bad?" needs no apology. The honest fold
> structure is harder, and reporting the hard number *is* the finding.

### ✏️ Assignment 3 — record what you measured

The cell above already assigned `RANDOM_CV_RMSE` and `BLOCKED_CV_RMSE`. The
grader checks that blocked came out **worse**. If it didn't, something is wrong
with your fold code — go back and find it rather than editing the numbers.

---
## Grade it

In [ ]:
grade_lab(3, globals())

### What you should be able to say out loud

- Rows are not independent — but **which** dependence you have is a measurement,
  not an assumption. Here the spatial one is ~0 and the repeated-measures one is
  **0.87**.
- Random CV cashes that dependence in as skill; blocked CV does not.
- Blocking the **corridor** rather than the row closes both at once.
- The gap over Surrey is small *because nothing has skill to inflate* — say that
  plainly rather than overclaiming.
- The real pipeline re-seeds the blocking 5 times, so no single lucky partition
  can carry the result (`experiment.blocked_cv`).